# One-Hot Encode Categorical Features

This notebook converts categorical features (amino acid letters) to numeric one-hot encoded columns.

## Why This Step?
After feature engineering, some columns contain amino acid letters (e.g., 'A', 'C', 'G'):
- `middle_aa`: Amino acid at position 15
- `cys_ctx_-2`, `cys_ctx_-1`, etc.: Context around cysteine

**Machine learning models need numbers, not letters!**

## Solution: One-Hot Encoding
```
middle_aa = 'A' → mid_A=1, mid_C=0, mid_D=0, ...
middle_aa = 'C' → mid_A=0, mid_C=1, mid_D=0, ...
```

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFY THESE
# ============================================================================

INPUT_FILE = '../data_engineered/train_with_features.csv'
OUTPUT_FILE = '../data_engineered/onehot/train_with_features_onehot.csv'
CHUNK_SIZE = 10000  # Rows per chunk

# Categorical columns to encode
CATEGORICAL_COLS = ['middle_aa', 'cys_ctx_-2', 'cys_ctx_-1', 'cys_ctx_1', 'cys_ctx_2']

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Chunk size: {CHUNK_SIZE:,}")

In [ ]:
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported")

## Step 1: Scan for All Unique Values

**Critical:** Before encoding, we must find ALL unique values across the entire dataset.

**Why?** Each chunk might have different values:
- Chunk 1: middle_aa = ['A', 'C', 'G']
- Chunk 2: middle_aa = ['D', 'E', 'F']

If we encode separately, they'll have different columns! We need consistent columns.

In [ ]:
print("Scanning for unique categorical values...\n")

unique_values = {col: set() for col in CATEGORICAL_COLS}

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    for col in CATEGORICAL_COLS:
        if col in chunk.columns:
            unique_values[col].update(chunk[col].unique())

print("Unique values found:")
for col, values in unique_values.items():
    print(f"  {col}: {len(values)} values → {sorted(values)}")

## Step 2: Encode Chunks with Consistent Columns

In [ ]:
print("\n" + "="*80)
print("ENCODING CATEGORICAL FEATURES")
print("="*80 + "\n")

first_chunk = True
chunk_num = 0
total_rows = sum(1 for _ in open(INPUT_FILE)) - 1

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    chunk_num += 1
    print(f"[Chunk {chunk_num}] Processing {len(chunk):,} rows...")
    
    # Separate metadata, labels, and features
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    metadata_cols = ['ID', 'Sequence'] + label_cols
    
    metadata = chunk[metadata_cols]
    features = chunk.drop(columns=metadata_cols)
    
    # One-hot encode each categorical column
    encoded_dfs = []
    
    for col in CATEGORICAL_COLS:
        if col in features.columns:
            # Create dummies
            dummies = pd.get_dummies(features[col], prefix=col)
            
            # Ensure ALL expected columns exist (even if value not in this chunk)
            for val in unique_values[col]:
                expected_col = f"{col}_{val}"
                if expected_col not in dummies.columns:
                    dummies[expected_col] = 0
            
            # Sort columns for consistency
            dummies = dummies[sorted(dummies.columns)]
            encoded_dfs.append(dummies)
            
            # Drop original column
            features = features.drop(columns=[col])
    
    # Combine all features
    if encoded_dfs:
        encoded_features = pd.concat([features] + encoded_dfs, axis=1)
    else:
        encoded_features = features
    
    # Combine with metadata
    final_chunk = pd.concat([metadata, encoded_features], axis=1)
    
    # Save
    if first_chunk:
        final_chunk.to_csv(OUTPUT_FILE, mode='w', index=False, header=True)
        first_chunk = False
        print(f"  Created {OUTPUT_FILE}")
        print(f"  Total columns: {len(final_chunk.columns):,}")
    else:
        final_chunk.to_csv(OUTPUT_FILE, mode='a', index=False, header=False)
    
    print(f"  Saved {len(final_chunk):,} rows")
    
    # Free memory
    del chunk, metadata, features, encoded_dfs, final_chunk
    gc.collect()

print("\n" + "="*80)
print("✓ ENCODING COMPLETE!")
print("="*80)

## Step 3: Verify Output

In [ ]:
import os

# Check output
df_check = pd.read_csv(OUTPUT_FILE, nrows=5)

print(f"\nOutput file: {OUTPUT_FILE}")
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(df_check.columns):,}")

# Check data types
print(f"\nData types:")
print(df_check.dtypes.value_counts())

# File size
file_size = os.path.getsize(OUTPUT_FILE)
print(f"\nFile size: {file_size/(1024*1024):.1f} MB")

print("\n✓ All features are now numeric and ready for ML models!")

df_check.head()

## Next Steps

1. **Create train/val split** → `04_train_val_split.ipynb`
2. **Train models** → RF and XGBoost

### What Changed:
- Before: `middle_aa = 'A'` (string)
- After: `mid_A=1, mid_C=0, mid_D=0, ...` (numbers)

### File Ready For:
- ✓ Train/validation splitting
- ✓ Random Forest training  
- ✓ XGBoost training
- ✓ All sklearn models